In [ ]:
%py
spark.catalog.setCurrentCatalog("purgo_databricks")

from pyspark.sql.functions import row_number  
from pyspark.sql.window import Window  
from pyspark.sql.types import StringType, StructType, StructField  

ROWS_SKIP = 3  # Number of rows to skip

def skip_first_n_rows(df, n):
    """
    Skips the first n rows of a DataFrame using DataFrame APIs and window functions.
    Args:
        df (DataFrame): Input DataFrame.
        n (int): Number of rows to skip.
    Returns:
        DataFrame: DataFrame after skipping the first n rows.
    Raises:
        ValueError: If n is not a non-negative integer.
    """
    if not isinstance(n, int) or n < 0:
        raise ValueError("rows_skip must be a non-negative integer")
    # Define a window specification for row numbering
    window_spec = Window.orderBy(lit(1))  # Use a constant column to preserve order
    # Add row_number column
    df_with_rownum = df.withColumn("row_num", row_number().over(window_spec))
    # Filter out the first n rows
    result_df = df_with_rownum.filter(df_with_rownum["row_num"] > n).select("value")
    return result_df

def validate_schema(df, expected_schema):
    """
    Validates that the DataFrame schema matches the expected schema (column name and type only).
    Args:
        df (DataFrame): DataFrame to validate.
        expected_schema (StructType): Expected schema.
    Returns:
        bool: True if schema matches, False otherwise.
    """
    actual_fields = [(f.name, type(f.dataType)) for f in df.schema.fields]
    expected_fields = [(f.name, type(f.dataType)) for f in expected_schema.fields]
    return actual_fields == expected_fields

from pyspark.sql.functions import lit  

try:
    # Read the source table as DataFrame
    source_df = spark.table("purgo_databricks.purgo_playground.actual_file")
    # Define expected schema for validation
    expected_schema = StructType([StructField("value", StringType(), True)])
    # Validate schema: column name and type only
    if not validate_schema(source_df, expected_schema):
        raise TypeError("Source DataFrame schema does not match expected schema")
    # Skip the first 3 rows using DataFrame API
    target_df = skip_first_n_rows(source_df, ROWS_SKIP)
    # Validate output schema
    if not validate_schema(target_df, expected_schema):
        raise TypeError("Result DataFrame schema does not match expected schema")
    # Output the resulting DataFrame (no logging)
    target_df.show(truncate=False)
except Exception as e:
    # Handle errors gracefully
    print(f"Error: {str(e)}")
# End of script
